In [1]:
import polars as pl
import time

## Big Data e Processamento Otimizado com Python II

### Manipulação do Parquet e StringCache

In [2]:
inicial = (
    pl.scan_parquet("../Aula12/202501_NovoBolsaFamilia_POLARS.parquet")
    .rename({
        'M�S COMPET�NCIA': 'MES_COMPETENCIA',
        'M�S REFER�NCIA': 'MES_REFERENCIA',
        'C�DIGO MUNIC�PIO SIAFI': 'CODIGO_MUNICIPIO_SIAFI',
        'NOME MUNIC�PIO': 'NOME_MUNICIPIO',
    })
    .with_columns(
        pl.col("VALOR PARCELA")
        .str.replace(",", ".")   
        .cast(pl.Float64)          
    )
)

display(inicial.collect())

MES_COMPETENCIA,MES_REFERENCIA,UF,CODIGO_MUNICIPIO_SIAFI,NOME_MUNICIPIO,CPF FAVORECIDO,NIS FAVORECIDO,NOME FAVORECIDO,VALOR PARCELA
i64,i64,str,i64,str,str,i64,str,f64
202501,202308,"""SP""",7071,"""SANTOS""","""***.085.106-**""",20643890445,"""FERNANDA RAMOS TEIXEIRA""",650.0
202501,202309,"""SP""",7071,"""SANTOS""","""***.085.106-**""",20643890445,"""FERNANDA RAMOS TEIXEIRA""",650.0
202501,202310,"""SP""",7071,"""SANTOS""","""***.085.106-**""",20643890445,"""FERNANDA RAMOS TEIXEIRA""",650.0
202501,202311,"""SP""",7071,"""SANTOS""","""***.085.106-**""",20643890445,"""FERNANDA RAMOS TEIXEIRA""",650.0
202501,202312,"""SP""",7071,"""SANTOS""","""***.085.106-**""",20643890445,"""FERNANDA RAMOS TEIXEIRA""",650.0
…,…,…,…,…,…,…,…,…
202501,202501,"""TO""",9643,"""XAMBIOA""","""""",16640890691,"""ZEIA DE SOUZA LUCIO""",750.0
202501,202501,"""TO""",9643,"""XAMBIOA""","""***.273.191-**""",20644881997,"""ZENILDE ALVES DOS SANTOS""",600.0
202501,202501,"""TO""",9643,"""XAMBIOA""","""""",19058661973,"""ZENOLIA RAMOS DA SILVA CARVALH…",600.0


### Filtros
Para selecionar linhas baseadas em condições lógicas (semelhante ao WHERE do SQL).

In [3]:
# Filtro Simples: Apenas beneficiários do Rio de Janeiro
q_rj = inicial.filter(pl.col("UF") == "RJ")

# Filtro Composto (E / OU): RJ e valor maior que 800
q_rj_alto = inicial.filter(
    (pl.col("UF") == "RJ") & (pl.col("VALOR PARCELA") > 800)
)

# Filtro com Lista (IS IN): Apenas estados do Sudeste
q_sudeste = inicial.filter(
    pl.col("UF").is_in(["SP", "RJ", "MG", "ES"])
)

display(q_rj.collect())
display(q_rj_alto.collect())
display(q_sudeste.collect())

MES_COMPETENCIA,MES_REFERENCIA,UF,CODIGO_MUNICIPIO_SIAFI,NOME_MUNICIPIO,CPF FAVORECIDO,NIS FAVORECIDO,NOME FAVORECIDO,VALOR PARCELA
i64,i64,str,i64,str,str,i64,str,f64
202501,202406,"""RJ""",5869,"""NOVA IGUACU""","""""",17009152797,"""PAULO DUARTE DOS SANTOS JUNIOR""",600.0
202501,202407,"""RJ""",5801,"""ANGRA DOS REIS""","""""",13907790986,"""CAMILA INACIO GOMES""",800.0
202501,202407,"""RJ""",5803,"""ARARUAMA""","""***.319.337-**""",16590291285,"""SIRLEA ALVES CORREA""",600.0
202501,202407,"""RJ""",5807,"""BARRA MANSA""","""""",21226891375,"""EMANUELLE ARAUJO DA SILVA""",750.0
202501,202407,"""RJ""",5807,"""BARRA MANSA""","""***.924.687-**""",13232334627,"""FABIANA PEREIRA DA SILVA VIEIR…",800.0
…,…,…,…,…,…,…,…,…
202501,202501,"""RJ""",5925,"""VOLTA REDONDA""","""""",13143785607,"""ZITIELEN APARECIDA CAETANO BIT…",650.0
202501,202501,"""RJ""",5925,"""VOLTA REDONDA""","""***.546.288-**""",23724063101,"""ZIULANDA NICOLAU SILVA""",300.0
202501,202501,"""RJ""",5925,"""VOLTA REDONDA""","""***.233.607-**""",20611706797,"""ZULEICA APARECIDA ALVES""",350.0


MES_COMPETENCIA,MES_REFERENCIA,UF,CODIGO_MUNICIPIO_SIAFI,NOME_MUNICIPIO,CPF FAVORECIDO,NIS FAVORECIDO,NOME FAVORECIDO,VALOR PARCELA
i64,i64,str,i64,str,str,i64,str,f64
202501,202407,"""RJ""",5813,"""CABO FRIO""","""***.820.616-**""",16204145798,"""CAMILA RAFAELA RAMOS SILVA""",1110.0
202501,202407,"""RJ""",5819,"""CAMPOS DOS GOYTACAZES""","""***.885.127-**""",16381234297,"""LUDIMILLA FERREIRA DE SOUZA GO…",1202.0
202501,202407,"""RJ""",5819,"""CAMPOS DOS GOYTACAZES""","""***.741.867-**""",16687809300,"""MYLENA MARTINS CORREA""",1076.0
202501,202407,"""RJ""",772,"""CARAPEBUS""","""***.382.557-**""",16169032082,"""BARBARA BORGES FERREIRA ANDRAD…",900.0
202501,202407,"""RJ""",5833,"""DUQUE DE CAXIAS""","""***.188.247-**""",20702439252,"""FLAVIA LEITE DE OLIVEIRA""",850.0
…,…,…,…,…,…,…,…,…
202501,202501,"""RJ""",5925,"""VOLTA REDONDA""","""""",13707804037,"""YURAIMA JOSEFINA ARREAZA ROCA""",950.0
202501,202501,"""RJ""",5925,"""VOLTA REDONDA""","""***.948.743-**""",13790540772,"""ZAILANE DOS SANTOS SILVA CAMAR…",850.0
202501,202501,"""RJ""",5925,"""VOLTA REDONDA""","""***.921.207-**""",12892010561,"""ZAINE ALI SEIFEDDINE""",900.0


MES_COMPETENCIA,MES_REFERENCIA,UF,CODIGO_MUNICIPIO_SIAFI,NOME_MUNICIPIO,CPF FAVORECIDO,NIS FAVORECIDO,NOME FAVORECIDO,VALOR PARCELA
i64,i64,str,i64,str,str,i64,str,f64
202501,202308,"""SP""",7071,"""SANTOS""","""***.085.106-**""",20643890445,"""FERNANDA RAMOS TEIXEIRA""",650.0
202501,202309,"""SP""",7071,"""SANTOS""","""***.085.106-**""",20643890445,"""FERNANDA RAMOS TEIXEIRA""",650.0
202501,202310,"""SP""",7071,"""SANTOS""","""***.085.106-**""",20643890445,"""FERNANDA RAMOS TEIXEIRA""",650.0
202501,202311,"""SP""",7071,"""SANTOS""","""***.085.106-**""",20643890445,"""FERNANDA RAMOS TEIXEIRA""",650.0
202501,202312,"""SP""",7071,"""SANTOS""","""***.085.106-**""",20643890445,"""FERNANDA RAMOS TEIXEIRA""",650.0
…,…,…,…,…,…,…,…,…
202501,202501,"""SP""",2973,"""ZACARIAS""","""""",20491700983,"""VANDERSON DIOGO DA SILVA""",600.0
202501,202501,"""SP""",2973,"""ZACARIAS""","""***.696.978-**""",16528891096,"""VANILDE ARMINDO DOS SANTOS""",600.0
202501,202501,"""SP""",2973,"""ZACARIAS""","""***.960.048-**""",16583160529,"""WELINGTON LUCAS BATISTA COELHO""",450.0


### Seleção e Exclusão
Para escolher ou remover colunas específicas.

In [4]:
# Selecionar apenas colunas de interesse
q_simples = inicial.select(["UF", "NOME_MUNICIPIO", "VALOR PARCELA"])

# .drop() remove as colunas listadas
q_sem_codigos = inicial.drop(["CODIGO_MUNICIPIO_SIAFI", "NIS FAVORECIDO"])

# Seleciona tudo (implícito) excluindo as colunas listadas
q_sem_codigos = inicial.select(pl.exclude(["CODIGO_MUNICIPIO_SIAFI", "NIS FAVORECIDO"]))

display(q_simples.collect())
display(q_sem_codigos.collect())

UF,NOME_MUNICIPIO,VALOR PARCELA
str,str,f64
"""SP""","""SANTOS""",650.0
"""SP""","""SANTOS""",650.0
"""SP""","""SANTOS""",650.0
"""SP""","""SANTOS""",650.0
"""SP""","""SANTOS""",650.0
…,…,…
"""TO""","""XAMBIOA""",750.0
"""TO""","""XAMBIOA""",600.0
"""TO""","""XAMBIOA""",600.0


MES_COMPETENCIA,MES_REFERENCIA,UF,NOME_MUNICIPIO,CPF FAVORECIDO,NOME FAVORECIDO,VALOR PARCELA
i64,i64,str,str,str,str,f64
202501,202308,"""SP""","""SANTOS""","""***.085.106-**""","""FERNANDA RAMOS TEIXEIRA""",650.0
202501,202309,"""SP""","""SANTOS""","""***.085.106-**""","""FERNANDA RAMOS TEIXEIRA""",650.0
202501,202310,"""SP""","""SANTOS""","""***.085.106-**""","""FERNANDA RAMOS TEIXEIRA""",650.0
202501,202311,"""SP""","""SANTOS""","""***.085.106-**""","""FERNANDA RAMOS TEIXEIRA""",650.0
202501,202312,"""SP""","""SANTOS""","""***.085.106-**""","""FERNANDA RAMOS TEIXEIRA""",650.0
…,…,…,…,…,…,…
202501,202501,"""TO""","""XAMBIOA""","""""","""ZEIA DE SOUZA LUCIO""",750.0
202501,202501,"""TO""","""XAMBIOA""","""***.273.191-**""","""ZENILDE ALVES DOS SANTOS""",600.0
202501,202501,"""TO""","""XAMBIOA""","""""","""ZENOLIA RAMOS DA SILVA CARVALH…",600.0


### Condicionais
Criar novas colunas com base em lógica (semelhante ao IF do Excel ou CASE WHEN do SQL).

In [5]:
# Categorizar o pagamento
q_categoria = inicial.with_columns(
    pl.when(pl.col("VALOR PARCELA") >= 900)
    .then(pl.lit("Alto Valor"))
    .when(pl.col("VALOR PARCELA") > 600)
    .then(pl.lit("Médio Valor"))
    .otherwise(pl.lit("Básico"))
    .alias("CATEGORIA_PAGAMENTO")
)

display(q_categoria.select(["VALOR PARCELA", "CATEGORIA_PAGAMENTO"]).collect())

VALOR PARCELA,CATEGORIA_PAGAMENTO
f64,str
650.0,"""Médio Valor"""
650.0,"""Médio Valor"""
650.0,"""Médio Valor"""
650.0,"""Médio Valor"""
650.0,"""Médio Valor"""
…,…
750.0,"""Médio Valor"""
600.0,"""Básico"""
600.0,"""Básico"""


In [6]:
# Partindo do LazyFrame 'q_categoria' já criado
q_analise_categoria = (
    q_categoria
    .group_by("CATEGORIA_PAGAMENTO")
    .agg([
        pl.len().alias("total_beneficiarios"),
        pl.col("VALOR PARCELA").mean().alias("media_pagamento"),
        pl.col("VALOR PARCELA").min().alias("menor_pagamento"),
        pl.col("VALOR PARCELA").max().alias("maior_pagamento"),
        pl.col("VALOR PARCELA").sum().alias("total_investido")
    ])
    .sort("media_pagamento", descending=True)
)

display(q_analise_categoria.collect())

CATEGORIA_PAGAMENTO,total_beneficiarios,media_pagamento,menor_pagamento,maior_pagamento,total_investido
str,u32,f64,f64,f64,f64
"""Alto Valor""",2103126,1038.537427,900.0,3938.0,2.1842e9
"""Médio Valor""",9657874,724.877575,601.0,898.0,7.0008e9
"""Básico""",8739422,520.723108,25.0,600.0,4.5508e9


### Fatiamento
Para pegar "pedaços" dos dados. No modo Lazy, o slice é muito eficiente pois o Polars lê apenas o necessário do disco.

In [7]:
# Pegar as primeiras 10 linhas
q_head = inicial.head(10)

# Pegar da linha 100 até a 150 (offset 100, length 50)
q_meio = inicial.slice(100, 50)

display(q_head.collect())
display(q_meio.collect())

MES_COMPETENCIA,MES_REFERENCIA,UF,CODIGO_MUNICIPIO_SIAFI,NOME_MUNICIPIO,CPF FAVORECIDO,NIS FAVORECIDO,NOME FAVORECIDO,VALOR PARCELA
i64,i64,str,i64,str,str,i64,str,f64
202501,202308,"""SP""",7071,"""SANTOS""","""***.085.106-**""",20643890445,"""FERNANDA RAMOS TEIXEIRA""",650.0
202501,202309,"""SP""",7071,"""SANTOS""","""***.085.106-**""",20643890445,"""FERNANDA RAMOS TEIXEIRA""",650.0
202501,202310,"""SP""",7071,"""SANTOS""","""***.085.106-**""",20643890445,"""FERNANDA RAMOS TEIXEIRA""",650.0
202501,202311,"""SP""",7071,"""SANTOS""","""***.085.106-**""",20643890445,"""FERNANDA RAMOS TEIXEIRA""",650.0
202501,202312,"""SP""",7071,"""SANTOS""","""***.085.106-**""",20643890445,"""FERNANDA RAMOS TEIXEIRA""",650.0
202501,202401,"""CE""",1389,"""FORTALEZA""","""***.854.493-**""",13024055192,"""IRENICE CAMELO VALETE""",750.0
202501,202401,"""PA""",479,"""LIMOEIRO DO AJURU""","""***.868.792-**""",16025132403,"""ANTONIA ELIZANA DA SILVA""",600.0
202501,202401,"""SP""",6313,"""CARAPICUIBA""","""***.972.798-**""",13310239852,"""DAMARIS DOS SANTOS OLIVEIRA""",800.0
202501,202401,"""SP""",6313,"""CARAPICUIBA""","""***.696.858-**""",20217442743,"""PRISCILA HELENA DE LIMA""",650.0


MES_COMPETENCIA,MES_REFERENCIA,UF,CODIGO_MUNICIPIO_SIAFI,NOME_MUNICIPIO,CPF FAVORECIDO,NIS FAVORECIDO,NOME FAVORECIDO,VALOR PARCELA
i64,i64,str,i64,str,str,i64,str,f64
202501,202406,"""MA""",937,"""TIMON""","""***.452.783-**""",16185066824,"""GISELIA DE ANDRADE LIMA""",650.0
202501,202406,"""MA""",949,"""VIANA""","""""",20927770940,"""MARINALDO NUNES COSTA""",600.0
202501,202406,"""MG""",4031,"""ALFENAS""","""***.970.326-**""",20345239495,"""TALYSSON DE SOUZA ESTEVAM""",600.0
202501,202406,"""MG""",4123,"""BELO HORIZONTE""","""***.510.066-**""",13035110092,"""CAMILA CUNHA DE OLIVEIRA""",700.0
202501,202406,"""MG""",4265,"""CARANGOLA""","""***.049.556-**""",16628504388,"""TAILANI FERREIRA""",600.0
…,…,…,…,…,…,…,…,…
202501,202406,"""RJ""",5869,"""NOVA IGUACU""","""""",17009152797,"""PAULO DUARTE DOS SANTOS JUNIOR""",600.0
202501,202406,"""RN""",1779,"""PARNAMIRIM""","""***.718.344-**""",12615870647,"""GEYSE MONTEIRO DA SILVA""",650.0
202501,202406,"""RO""",3,"""PORTO VELHO""","""***.375.102-**""",20345823987,"""NELSON GARCIA SOBRINHO""",600.0


In [8]:
# Recorte de linhas specíficas (ex: linha 0, 10 e 50)
q_recorte_indices = (
    inicial
    .with_row_index("id_linha")  # Cria uma coluna temporária com o número da linha
    .filter(pl.col("id_linha").is_in([0, 10, 50])) # Filtra pelos números desejados
    .select(["UF", "NOME_MUNICIPIO"]) # Seleciona as colunas
)

display(q_recorte_indices.collect())

UF,NOME_MUNICIPIO
str,str
"""SP""","""SANTOS"""
"""SP""","""SANTOS"""
"""PR""","""GUARAPUAVA"""


### Agrupamento Avançado (group_by + agg)


In [9]:
q_stats_mun = (
    inicial.group_by("NOME_MUNICIPIO")
    .agg([
        pl.len().alias("total_familias"),  # Contagem rápida
        pl.col("VALOR PARCELA").sum().alias("total_repassado"),
        pl.col("VALOR PARCELA").max().alias("maior_pagamento"),
        pl.col("UF").first().alias("uf_origem") # Mantém a UF do municipio
    ])
    .sort("total_repassado", descending=True)
)

display(q_stats_mun.collect())

NOME_MUNICIPIO,total_familias,total_repassado,maior_pagamento,uf_origem
str,u32,f64,f64,str
"""SAO PAULO""",679740,4.4679095e8,3322.0,"""SP"""
"""RIO DE JANEIRO""",499084,3.27039464e8,3072.0,"""RJ"""
"""FORTALEZA""",324273,2.09478569e8,2562.0,"""CE"""
"""SALVADOR""",299582,1.93376165e8,2804.0,"""BA"""
"""MANAUS""",261304,1.80283025e8,2996.0,"""AM"""
…,…,…,…,…
"""MONTAURI""",11,6650.0,1000.0,"""RS"""
"""CORONEL PILAR""",9,6469.0,1444.0,"""RS"""
"""BRACO DO TROMBUDO""",13,6287.0,651.0,"""SC"""


### StringCache e Otimização de CPU

Como estamos na fase de pré-processamento de Big Data, precisamos conectar o conceito de hardware com o código.

#### O Problema da String: 

Textos são "pesadas" para a CPU processar. Comparar a palavra "RIO DE JANEIRO" com "RIO DE JANEIRO" exige verificar letra por letra. Isso ocupa muito espaço na memória Cache (L1/L2/L3) do processador, que é ultrarrápida mas muito pequena.

Uma solução? Em vez de guardar a palavra "SP" milhões de vezes, o computador cria um dicionário: 0 = SP, 1 = RJ. Ele processa números (inteiros), o que é mais rápido para a CPU.

O **StringCache** do Polars: Garante que esse dicionário seja GLOBAL.

- *Sem StringCache*: Se você tiver dois DataFrames, "SP" pode ser o código 0 no primeiro e o código 5 no segundo. Isso impede junções (joins) corretas sem recriar os dados.

- *Com StringCache*: "SP" será o código 0 em qualquer operação enquanto o cache estiver ativo.

#### Vantagens Técnicas:

Uso de Memória RAM: Em colunas com *baixa cardinalidade* (muita repetição, como UF ou NOME_MUNICIPIO), o uso de memória cai drasticamente.

Velocidade em Joins e Agrupamentos: Agrupar por inteiros é muito mais rápido do que por strings para a lógica computacional.

In [10]:
print("=== TESTE 1: Processamento SEM StringCache ===")
start_1 = time.time()

q_sem_cache = (
    pl.scan_parquet("../Aula12/202501_NovoBolsaFamilia_POLARS.parquet")
    .rename({
        'M�S COMPET�NCIA': 'MES_COMPETENCIA',
        'M�S REFER�NCIA': 'MES_REFERENCIA',
        'C�DIGO MUNIC�PIO SIAFI': 'CODIGO_MUNICIPIO_SIAFI',
        'NOME MUNIC�PIO': 'NOME_MUNICIPIO',
    })
    .with_columns(
        pl.col("VALOR PARCELA")
        .str.replace(",", ".")   
        .cast(pl.Float64)          
    )
    .group_by(["UF", "NOME_MUNICIPIO"]) 
    .agg([
        pl.col("VALOR PARCELA").mean().alias("media"),
        pl.col("VALOR PARCELA").std().alias("desvio")
    ])
    .sort("media", descending=True)
)

df_1 = q_sem_cache.collect()
end_1 = time.time()
tempo_sem = end_1 - start_1
print(f"Tempo SEM Cache: {tempo_sem:.4f} segundos")
print(f"Uso de memória estimado (coluna UF): {df_1['UF'].estimated_size()} bytes\n")

=== TESTE 1: Processamento SEM StringCache ===
Tempo SEM Cache: 0.9173 segundos
Uso de memória estimado (coluna UF): 11140 bytes



In [11]:
print("=== TESTE 2: Processamento COM StringCache ===")
# O StringCache funciona como um contexto (with)
with pl.StringCache():
    start_2 = time.time()
    
    q_com_cache = (
    pl.scan_parquet("../Aula12/202501_NovoBolsaFamilia_POLARS.parquet")
    .rename({
        'M�S COMPET�NCIA': 'MES_COMPETENCIA',
        'M�S REFER�NCIA': 'MES_REFERENCIA',
        'C�DIGO MUNIC�PIO SIAFI': 'CODIGO_MUNICIPIO_SIAFI',
        'NOME MUNIC�PIO': 'NOME_MUNICIPIO',
    })
    .with_columns(
        pl.col("VALOR PARCELA")
        .str.replace(",", ".")   
        .cast(pl.Float64)          
    )
    .group_by(["UF", "NOME_MUNICIPIO"]) 
    .agg([
        pl.col("VALOR PARCELA").mean().alias("media"),
        pl.col("VALOR PARCELA").std().alias("desvio")
    ])
    .sort("media", descending=True)
)
    
    df_2 = q_com_cache.collect() # Ação!
    end_2 = time.time()

tempo_com = end_2 - start_2
print(f"Tempo COM Cache: {tempo_com:.4f} segundos")
print(f"Uso de memória estimado (coluna UF): {df_2['UF'].estimated_size()} bytes\n")

=== TESTE 2: Processamento COM StringCache ===
Tempo COM Cache: 1.0065 segundos
Uso de memória estimado (coluna UF): 11140 bytes

